In [1]:
import pandas as pd

dim_customer = pd.read_csv("data/star_schema/Dim_Customer.csv")
print("Columns before fix:", list(dim_customer.columns))

# drop duplicate columns, keeping the first occurrence of each name
dim_customer = dim_customer.loc[:, ~dim_customer.columns.duplicated()]

print("Columns after fix:", list(dim_customer.columns))
dim_customer.to_csv("data/star_schema/Dim_Customer.csv", index=False)

Columns before fix: ['CustomerID', 'Age', 'Gender', 'Region', 'AcquisitionChannel', 'SignupDate', 'RFM_Score', 'CustomerSegment', 'Tenure', 'ChurnProbability_x', 'ChurnFlag_x', 'PredictedCLTV_x', 'IsScored_x', 'ChurnProbability_y', 'ChurnFlag_y', 'PredictedCLTV_y', 'IsScored_y']
Columns after fix: ['CustomerID', 'Age', 'Gender', 'Region', 'AcquisitionChannel', 'SignupDate', 'RFM_Score', 'CustomerSegment', 'Tenure', 'ChurnProbability_x', 'ChurnFlag_x', 'PredictedCLTV_x', 'IsScored_x', 'ChurnProbability_y', 'ChurnFlag_y', 'PredictedCLTV_y', 'IsScored_y']


In [2]:
dim_customer = pd.read_csv("data/star_schema/Dim_Customer.csv")

cols = ["ChurnProbability", "ChurnFlag", "PredictedCLTV", "IsScored"]

# Verify _x and _y are actually identical before we throw one away
all_match = True
for col in cols:
    x_col, y_col = f"{col}_x", f"{col}_y"
    mismatches = (dim_customer[x_col] != dim_customer[y_col]) & ~(
        dim_customer[x_col].isna() & dim_customer[y_col].isna()
    )
    n_mismatch = mismatches.sum()
    if n_mismatch > 0:
        all_match = False
        print(f"WARNING: {col}_x and {col}_y differ in {n_mismatch} rows! "
              f"Do not proceed blindly — investigate which one is correct.")
    else:
        print(f"{col}: _x and _y match exactly. Safe to drop the duplicate.")

if all_match:
    # keep the _x version (doesn't matter which, they're identical), rename it back, drop _y
    rename_map = {f"{col}_x": col for col in cols}
    drop_cols = [f"{col}_y" for col in cols]
    dim_customer = dim_customer.drop(columns=drop_cols).rename(columns=rename_map)
    dim_customer.to_csv("data/star_schema/Dim_Customer.csv", index=False)
    print("\nFixed. Final columns:", list(dim_customer.columns))
else:
    print("\nNOT saved — fix the mismatch above before re-running.")

ChurnProbability: _x and _y match exactly. Safe to drop the duplicate.
ChurnFlag: _x and _y match exactly. Safe to drop the duplicate.
PredictedCLTV: _x and _y match exactly. Safe to drop the duplicate.
IsScored: _x and _y match exactly. Safe to drop the duplicate.

Fixed. Final columns: ['CustomerID', 'Age', 'Gender', 'Region', 'AcquisitionChannel', 'SignupDate', 'RFM_Score', 'CustomerSegment', 'Tenure', 'ChurnProbability', 'ChurnFlag', 'PredictedCLTV', 'IsScored']
